## AFT Model Mathematics

The **Attention-Free Transformer (AFT)** replaces the conventional softmax attention mechanism with a learned positional bias and elementwise gating. Given an input sequence

$$
X \in \mathbb{R}^{B \times T \times D},
$$

the query, key, and value projections are computed as

$$
Q = W_qX, \qquad
K = W_kX, \qquad
V = W_vX.
$$

Let

$$
W^{b} \in \mathbb{R}^{T \times T}
$$

be the learnable positional bias matrix.

To improve numerical stability, the maximum key value is subtracted before exponentiation:

$$
\widetilde{K}
=
\exp\left(
K - \max_{t}(K_t)
\right).
$$

The weighted value representation is then computed as

$$
\mathrm{weighted}(t)
=
\frac{
\exp(W^{b})
\left(
\widetilde{K}\odot V
\right)
}{
\exp(W^{b})\,\widetilde{K}
+
\varepsilon
},
$$

where

- $\odot$ denotes elementwise multiplication.
- The division is performed elementwise.
- $\varepsilon$ is a small constant added to avoid division by zero.

The output is then gated using the sigmoid activation applied to the query:

$$
Y
=
\sigma(Q)
\odot
\mathrm{weighted}.
$$

Finally, the output is projected back to the model dimension:

$$
\mathrm{output}
=
W_oY.
$$

This corresponds to the implementation by:

1. Computing the query, key, and value projections.
2. Subtracting the maximum value from the keys for numerical stability.
3. Exponentiating the stabilized keys.
4. Applying the learned positional bias via matrix multiplication.
5. Combining the result with the value vectors using elementwise multiplication.
6. Normalizing by the biased key sum.
7. Gating the normalized output using $\sigma(Q)$.
8. Applying the final linear projection using $W_o$.

In [2]:
import torch
import math
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

In [3]:
# =============================================================================
# 1. AFT MODEL IMPLEMENTATION (STABILIZED)
# =============================================================================

class AFTFull(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim=64):
        super().__init__()
        self.dim = dim
        self.hidden_dim = hidden_dim
        self.to_q = nn.Linear(dim, hidden_dim)
        self.to_k = nn.Linear(dim, hidden_dim)
        self.to_v = nn.Linear(dim, hidden_dim)
        self.project = nn.Linear(hidden_dim, dim)
        self.wbias = nn.Parameter(torch.Tensor(max_seqlen, max_seqlen))
        nn.init.xavier_uniform_(self.wbias)

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.to_q(x)
        K = self.to_k(x)
        V = self.to_v(x)
        
        temp_wbias = self.wbias[:T, :T].unsqueeze(0) 
        Q_sig = torch.sigmoid(Q)
        
        # Subtract max for numerical stability against exp overflow
        K_stable = K - K.max(dim=1, keepdim=True)[0]
        exp_K = torch.exp(K_stable)
        
        temp = torch.exp(temp_wbias) @ (exp_K * V)
        weighted = temp / (torch.exp(temp_wbias) @ exp_K + 1e-6)
        Yt = Q_sig * weighted

        return self.project(Yt)


In [4]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dp=0.1):
        super().__init__()
        self.l1 = nn.Linear(dim, hidden_dim)
        self.g1 = nn.GELU()
        self.l2 = nn.Linear(hidden_dim, dim)
        self.d1 = nn.Dropout(dp)

    def forward(self, x):
        x = self.l1(x)
        x = self.g1(x)
        x = self.d1(x)
        return self.l2(x)      

In [5]:
class AFTEncoderBlock(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim, p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.attn = AFTFull(max_seqlen, dim, hidden_dim)
        self.mlp = MLP(dim, hidden_dim, dp=p)
        self.d1 = nn.Dropout(p)
        self.d2 = nn.Dropout(p)

    def forward(self, x):
        x_norm = self.ln1(x)
        x = x + self.d1(self.attn(x_norm))
        x_norm = self.ln2(x)
        out = x + self.d2(self.mlp(x_norm))
        return out

In [6]:
class AFT(nn.Module):
    def __init__(self, vocab_size, max_seqlen, dim, hidden_dim, depth=4, p=0.1):
        super().__init__()
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)
        
        # Simple learnable absolute positional embeddings
        self.pos_embed = nn.Embedding(max_seqlen, dim)
        
        self.enc = nn.Sequential(*[
            AFTEncoderBlock(max_seqlen, dim, hidden_dim, p=p) 
            for _ in range(depth)
        ])
        self.dec = nn.Linear(dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        device = x.device
        
        positions = torch.arange(0, T, device=device).unsqueeze(0).expand(B, T)
        x = self.embed(x) * math.sqrt(self.dim) + self.pos_embed(positions)
        
        x = self.enc(x)
        out = self.dec(x)
        return out


In [7]:
# =============================================================================
# 2. OPUS DATASET PIPELINE
# =============================================================================

class OpusTextDataset(Dataset):
    def __init__(self, max_seqlen, tokenizer_name="gpt2"):
        print("Loading OPUS Books dataset via Hugging Face...")
        # Loading English-French books context; we'll train language modeling on the English portion
        raw_dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")
        
        print("Initializing Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        # Ensure standard padding token is handled
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.max_seqlen = max_seqlen
        self.input_ids = []

        print("Tokenizing real text corpus...")
        # Process and pack sentences into fixed-length segments
        buffer = []
        for item in raw_dataset:
            text = item['translation']['en']
            tokens = self.tokenizer.encode(text)
            buffer.extend(tokens)
            
            # Chunk long continuous stream into discrete sequence blocks
            while len(buffer) >= (max_seqlen + 1):
                self.input_ids.append(buffer[:max_seqlen + 1])
                buffer = buffer[max_seqlen:]
                
            # Limit dataset size for a fast verification run
            if len(self.input_ids) >= 1500:
                break

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Convert chunk into tensor
        chunk = torch.tensor(self.input_ids[idx], dtype=torch.long)
        # Source sequence: tokens 0 to N-1
        x = chunk[:-1]
        # Target sequence (Next Token Prediction): tokens 1 to N
        y = chunk[1:]
        return x, y

    def get_vocab_size(self):
        return self.tokenizer.vocab_size


In [8]:
# =============================================================================
# 3. TRAINING ROUTINE
# =============================================================================

def train_on_opus():
    # Hyperparameters
    MAX_SEQLEN = 64
    EMBED_DIM = 512
    HIDDEN_DIM = 512
    DEPTH = 100
    BATCH_SIZE = 16
    EPOCHS = 10
    LR = 5e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Prepare Dataset & Loader
    dataset = OpusTextDataset(max_seqlen=MAX_SEQLEN, tokenizer_name="gpt2")
    VOCAB_SIZE = dataset.get_vocab_size()
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\nConfiguration Details:")
    print(f"-> Device: {DEVICE}")
    print(f"-> Dataset Samples: {len(dataset)}")
    print(f"-> Model Vocabulary Size: {VOCAB_SIZE}")
    print("Initializing AFT Model Architecture...")

    # Build Model
    model = AFT(
        vocab_size=VOCAB_SIZE, 
        max_seqlen=MAX_SEQLEN, 
        dim=EMBED_DIM, 
        hidden_dim=HIDDEN_DIM, 
        depth=DEPTH
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    print("\nBeginning Training Pipeline...")
    model.train()
    
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        correct_tokens = 0
        total_tokens = 0
        
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            
            # Forward execution
            outputs = model(inputs) # Shape: (B, T, Vocab_Size)
            
            # Calculate cross entropy over execution matrix dimensions
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Evaluation metrics metrics
            epoch_loss += loss.item()
            preds = outputs.argmax(dim=-1)
            correct_tokens += (preds == targets).sum().item()
            total_tokens += targets.numel()
            
            if batch_idx % 20 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(dataloader)} | Current Batch Loss: {loss.item():.4f}")
                
        avg_loss = epoch_loss / len(dataloader)
        accuracy = (correct_tokens / total_tokens) * 100
        print(f"=== Epoch {epoch+1:02d} Complete === | Average Loss: {avg_loss:.4f} | Token Accuracy: {accuracy:.2f}%\n")

if __name__ == '__main__':
    train_on_opus()

Loading OPUS Books dataset via Hugging Face...


README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Initializing Tokenizer...


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing real text corpus...

Configuration Details:
-> Device: cuda
-> Dataset Samples: 1500
-> Model Vocabulary Size: 50257
Initializing AFT Model Architecture...

Beginning Training Pipeline...
Epoch 1/10 | Batch 0/94 | Current Batch Loss: 55.9262
Epoch 1/10 | Batch 20/94 | Current Batch Loss: 23.8149
Epoch 1/10 | Batch 40/94 | Current Batch Loss: 18.3379
Epoch 1/10 | Batch 60/94 | Current Batch Loss: 14.5401
Epoch 1/10 | Batch 80/94 | Current Batch Loss: 13.1091
=== Epoch 01 Complete === | Average Loss: 19.8141 | Token Accuracy: 10.74%

Epoch 2/10 | Batch 0/94 | Current Batch Loss: 8.4758
Epoch 2/10 | Batch 20/94 | Current Batch Loss: 8.3509
Epoch 2/10 | Batch 40/94 | Current Batch Loss: 8.5634
Epoch 2/10 | Batch 60/94 | Current Batch Loss: 7.9865
Epoch 2/10 | Batch 80/94 | Current Batch Loss: 8.4779
=== Epoch 02 Complete === | Average Loss: 7.9622 | Token Accuracy: 18.38%

Epoch 3/10 | Batch 0/94 | Current Batch Loss: 5.4392
Epoch 3/10 | Batch 20/94 | Current Batch Loss: 5.4029
